In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torchvision import datasets, transforms

# Part 1: Dataset Understanding and Inspection

Load the dataset and check the **class distribution per split**.

In [ ]:
data_dir = Path("/kaggle/input/competitions/artificial-intelligencefor-histopathology/Histo images")

train_dir = data_dir / "train"
val_dir = data_dir / "val"
test_dir = data_dir / "test"

for split in [train_dir, val_dir, test_dir]:
    print(f"\n{split.name} classes:")
    for cls in os.listdir(split):
        n = len(os.listdir(split / cls))
        print(f"{cls}: {n} images")

In [ ]:
def show_random_images(folder, n=6):
    #TODO
    ...



## Create Datasets

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    
    transforms.Resize((224,224)),
    
    transforms.ToTensor(),
    
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
from torchvision import datasets

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

from PIL import Image

def remove_corrupted(dataset):

    valid_samples = []

    for path, label in dataset.samples:

        try:
            img = Image.open(path)
            img.verify()   # check image integrity
            valid_samples.append((path, label))

        except:
            print("Removing corrupted image:", path)

    dataset.samples = valid_samples
    dataset.imgs = valid_samples

remove_corrupted(train_dataset)
remove_corrupted(val_dataset)
remove_corrupted(test_dataset)

In [ ]:
from torch.utils.data import DataLoader

batch_size = ...

train_loader = DataLoader(
   ...
)

val_loader = DataLoader(
    ...
)

test_loader = DataLoader(
    ...
)

In [ ]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First labels:", labels[:10])

# Part 2: Baseline classifier with handcrafted features

Define handmade features


In [ ]:
import numpy as np

def extract_features(img):
    ...
    return features

In [ ]:
X_train = []
y_train = []

for i in range(len(train_dataset)):

    try:
        img, label = train_dataset[i]

        img = img.permute(1,2,0).numpy()
        img = (img*255).astype(np.uint8)

        features = extract_features(img)

        X_train.append(features)
        y_train.append(label)

    except:
        continue

X_train = np.array(X_train)
y_train = np.array(y_train)

print("Train feature matrix:", X_train.shape)

In [ ]:
X_test = []
y_test = []

for i in range(len(test_dataset)):

    try:
        img, label = test_dataset[i]

        img = img.permute(1,2,0).numpy()
        img = (img*255).astype(np.uint8)

        features = extract_features(img)

        X_test.append(features)
        y_test.append(label)

    except:
        continue

X_test = np.array(X_test)
y_test = np.array(y_test)

print("Test feature matrix:", X_test.shape)

In [ ]:
X_val = []
y_val = []

for i in range(len(val_dataset)):

    try:
        img, label = val_dataset[i]

        img = img.permute(1,2,0).numpy()
        img = (img*255).astype(np.uint8)

        features = extract_features(img)

        X_val.append(features)
        y_val.append(label)

    except:
        continue

X_val = np.array(X_val)
y_val = np.array(y_val)

print("Validation feature matrix:", X_val.shape)

In [ ]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_data = TensorDataset(X_train, y_train)
val_data = TensorDataset(X_val, y_val)
test_data = TensorDataset(X_test, y_test)

from PIL import Image

def remove_corrupted(dataset):

    valid_samples = []

    for path, label in dataset.samples:

        try:
            img = Image.open(path)
            img.verify()   # check image integrity
            valid_samples.append((path, label))

        except:
            print("Removing corrupted image:", path)

    dataset.samples = valid_samples
    dataset.imgs = valid_samples

remove_corrupted(train_dataset)
remove_corrupted(val_dataset)
remove_corrupted(test_dataset)


train_loader_features = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader_features = DataLoader(val_data, batch_size=64)
test_loader_features = DataLoader(test_data, batch_size=64,shuffle=False)

Define a classifier

In [ ]:
import torch.nn as nn

class FeatureClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            ...
        )

    def forward(self,x):

        return self.net(x)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = FeatureClassifier().to(device)

In [ ]:
criterion = ...

optimizer = ...

In [ ]:
epochs = 50

for epoch in range(epochs):

   ...

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for x, y in val_loader_features:

        x = x.to(device)
        y = y.to(device)

        outputs = model(x)

        preds = torch.argmax(outputs, dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

accuracy = correct / total

print("Validation accuracy:", accuracy)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score

def collect_preds(model, loader, device):
    all_preds, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_preds)

def cm_with_percent_labels(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    cm_percent = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    labels = np.array([
        f"{cm[i,j]}\n{cm_percent[i,j]:.1f}%"
        for i in range(n_classes)
        for j in range(n_classes)
    ]).reshape(n_classes, n_classes)

    return cm, labels

def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

In [ ]:
class_names = train_dataset.classes
n_classes = len(class_names)

y_val_true, y_val_pred = collect_preds(model, val_loader_features, device)
y_test_true, y_test_pred = collect_preds(model, test_loader_features, device)

In [ ]:
cm_val, labels_val = cm_with_percent_labels(y_val_true, y_val_pred, n_classes)
cm_test, labels_test = cm_with_percent_labels(y_test_true, y_test_pred, n_classes)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(
    cm_val,
    annot=labels_val,
    fmt="",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[0]
)
axes[0].set_title("Validation Confusion Matrix (count + %)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(
    cm_test,
    annot=labels_test,
    fmt="",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[1]
)
axes[1].set_title("Test Confusion Matrix (count + %)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()

In [ ]:
val_metrics = compute_metrics(y_val_true, y_val_pred)
test_metrics = compute_metrics(y_test_true, y_test_pred)

print("{:<18} {:>12} {:>12}".format("Metric", "VAL", "TEST"))
print("-"*44)
for k in val_metrics.keys():
    print("{:<18} {:>12.4f} {:>12.4f}".format(k, val_metrics[k], test_metrics[k]))

# Part 3:  Deep Feature Extraction with a CNN

In [ ]:
import torchvision.models as models
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.resnet50()

ckpt_path = "/kaggle/input/competitions/artificial-intelligencefor-histopathology/Histo images/best_ckpt.pth"

state_dict = torch.load(ckpt_path, map_location=device)

model.load_state_dict(state_dict, strict=False)

model = model.to(device)

model.eval()

In [ ]:
import torch.nn as nn

feature_extractor = ...

feature_extractor.eval()

In [ ]:
def extract_deep_features(loader):

    ...

    return features, labels

In [ ]:
X_train_deep, y_train_deep = extract_deep_features(train_loader)
X_val_deep, y_val_deep = extract_deep_features(val_loader)
X_test_deep, y_test_deep = extract_deep_features(test_loader)

print(X_train_deep.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_data_deep = TensorDataset(X_train_deep, y_train_deep)
val_data_deep = TensorDataset(X_val_deep, y_val_deep)
test_data_deep = TensorDataset(X_test_deep, y_test_deep)

train_loader_deep = DataLoader(train_data_deep, batch_size=64, shuffle=True)
val_loader_deep = DataLoader(val_data_deep, batch_size=64)
test_loader_deep = DataLoader(test_data_deep, batch_size=64)

In [ ]:
class DeepClassifier(nn.Module):
    ...

In [ ]:
model_deep = DeepClassifier().to(device)

criterion = ...

optimizer = ...

In [ ]:
epochs = 20

for epoch in range(epochs):

    ...

In [ ]:
class_names = train_dataset.classes
n_classes = len(class_names)

y_val_true, y_val_pred = collect_preds(model_deep, val_loader_deep, device)
y_test_true, y_test_pred = collect_preds(model_deep, test_loader_deep, device)

cm_val, labels_val = cm_with_percent_labels(y_val_true, y_val_pred, n_classes)
cm_test, labels_test = cm_with_percent_labels(y_test_true, y_test_pred, n_classes)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(
    cm_val,
    annot=labels_val,
    fmt="",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[0]
)

axes[0].set_title("Validation Confusion Matrix (count + %)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")


sns.heatmap(
    cm_test,
    annot=labels_test,
    fmt="",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[1]
)

axes[1].set_title("Test Confusion Matrix (count + %)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()